# Compare `calc_jacobian` vs `calc_jacobian_updatedR` (`Rfree`)

Same parameters as `DurableModel_Combined_Inf_Test.ipynb`:
$S=0.2$, $\sigma_p=0.1$, `PriceShkCount=5`, `sigma_eps=0`.

## What differs

Both methods are Auclert et al. fake-news Jacobians of aggregates $(C, DEX, TEX, D, A, ADJ)$ w.r.t. a one-period `Rfree` impulse. They share the same finite-horizon **policy** solve: at the shocked date, $a',c,d$ come from the Bellman at $R_{ss}+dx$. They differ only in the **histogram** that moves the distribution,

$$
m' = \frac{R\,a'}{\psi} + \theta, \qquad n' = \frac{(1-\delta)d'}{\psi}.
$$

- **Baseline** (`calc_jacobian` / `calc_jacobian_cond`): $\Lambda_t$ always uses $R_{ss}$.
- **updatedR**: $\Lambda_t$ uses that date's $R_t$ (the same $R$ the Bellman used when choosing $a'_t$).

So the household saves as if the return is $R_{ss}+dx$, but only updatedR actually credits that return when mapping $a'$ into next $m$.

## Math

Write $m'(R,a'(R)) = R\,a'(R)+\theta$. Then

$$
\frac{dm'}{dR} = \underbrace{a'(R)}_{\text{mechanical}} + \underbrace{R\,\frac{da'}{dR}}_{\text{policy}}.
$$

Baseline curly $D$ (one-step distribution deviation) contains only the policy term. updatedR contains both. Curly $Y$ (policy evaluated on the SS distribution) is the same in both, so news columns of $J_C$ that are mostly curly $Y$ move little; $J_A$ and column $s=0$ (which is distribution-only) move first.

HARK timing: period-$t$ `Rfree` is the return from $t$ to $t+1$. Consistency requires $\Lambda_t$ to use that same $R_t$.

## Code

`_solve_perturbed_finite` sets `Rfree = path[t]` for the Bellman, then **restores** $R_{ss}$ in a `finally` block. `calc_transition_matrix_dur` then builds

```python
bNext = R_vec * aPol - costs    # R_vec from self.Rfree (SS) unless param_path is passed
```

- Baseline calls `calc_transition_matrix_dur()` with no path $\Rightarrow$ $R_{ss}$ in every $\Lambda_t$.
- updatedR passes `param_path={Rfree: path}` so period $k$ uses `path[k]`.
- Cond: `_search_on_dist_grid` also restores $R$ before `_tran_matrix_from_pols`. updatedR_cond sets `use_period_R=True` so that builder gets period-$t$ $R$ explicitly.

`labor` Jacobians are unaffected: income already enters $\Lambda$ via `IncShkDstn_path`. `DiscFac` likewise never appears in `bNext`.


In [1]:
import copy
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from DurableModel import DurableConsumer

JAC_KEYS = ["C", "DEX", "TEX", "D", "A", "ADJ"]
T = 50

durable_defaults = dict(
    CRRA=2.0,
    DiscFac=0.975,
    LivPrb=1.0,
    alpha=0.75,
    d_ubar=1e-4,
    dDepr=0.05,
    adjC=0.10,
    MaintCost=0.0,
    OpCost=0.0,
    Rfree=1.04**(0.25),
    RSpread=0.0,
    PermGroFac=1.0,
    LTV=0.0,
    BoroCnstArt=0.0,
    PermShkStd=0.0,
    PermShkCount=1,
    TranShkStd=0.2,
    TranShkCount=7,
    UnempPrb=0.0,
    IncUnemp=0.0,
    IncUnemp_Std=0.1,
    truncation=2,
    taxrate=0.0,
    labor=1.0,
    wage=1.0,
    sigma_eps=0.0,
    SearchCost=0.1,
    ScaleBool=False,
    e_grid_count=101,
    PriceShkCount=5,
    PriceShkStd=0.1,
    use_expected=True,
    use_TransitionMatrix=False,
    nNrmMax=10,
    nNrmCount=200,
    mNrmMax=16.0,
    mNrmCount=200,
    xNrmMax=26,
    xNrmCount=100,
    aNrmMax=16.0,
    aNrmCount=200,
    NestFac=1,
    grid_type="exp_mult",
    TranMatrix_mCount=1010,
    TranMatrix_bCount=1000,
    N_scan=800,
    cycles=0,
    T_cycle=1,
    tol=1e-8,
    max_cycles=500,
    min_cycles=200,
    distance_metric="mse",
    verbose=False,
)

params = copy.deepcopy(durable_defaults)
params["sigma_eps"] = 0.0
params["SearchCost"] = 0.2
params["PriceShkStd"] = 0.1
params["PriceShkCount"] = 5
params["TranMatrixmCount"] = 250
params["TranMatrixmMax"] = durable_defaults["mNrmMax"]
params["TranMatrixnMax"] = durable_defaults["nNrmMax"]
params["diagnostics"] = False


def pack_jac(tup):
    return dict(zip(JAC_KEYS, tup))


def jac_diff_table(J0, J1):
    rows = []
    for k in JAC_KEYS:
        d = J1[k] - J0[k]
        rows.append(dict(
            var=k,
            max_abs=float(np.max(np.abs(d))),
            max_abs_col0=float(np.max(np.abs(d[:, 0]))),
            rms=float(np.sqrt(np.mean(d**2))),
        ))
    return pd.DataFrame(rows).set_index("var")


def plot_jac_triplet(J0, J1, title):
    fig, axes = plt.subplots(len(JAC_KEYS), 3, figsize=(12, 16))
    col_titles = [r"baseline $\Lambda(R_{ss})$", r"updatedR $\Lambda(R_t)$", "difference"]
    for i, name in enumerate(JAC_KEYS):
        mats = [J0[name], J1[name], J1[name] - J0[name]]
        for j, (ax, M) in enumerate(zip(axes[i], mats)):
            lim = np.nanmax(np.abs(M)) or 1.0
            im = ax.imshow(M, origin="upper", cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="auto")
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            if i == 0:
                ax.set_title(col_titles[j])
            if j == 0:
                ax.set_ylabel(name)
            ax.set_xlabel("s")
    fig.suptitle(title, y=1.01)
    fig.tight_layout()
    plt.show()

    fig, axes = plt.subplots(2, 3, figsize=(12, 6))
    for ax, name in zip(axes.flat, JAC_KEYS):
        ax.plot(J0[name][:, 0], "k-", label="baseline")
        ax.plot(J1[name][:, 0], "r-", label="updatedR")
        ax.plot(J1[name][:, 0] - J0[name][:, 0], "k--", label="difference")
        ax.axhline(0.0, color="k", lw=0.5)
        ax.set_title(name)
        ax.set_xlabel("t")
        ax.grid(color="k", linestyle=":", linewidth=0.5)
        ax.legend(fontsize=8)
    fig.suptitle(title + r"  (column $s=0$)")
    fig.tight_layout()
    plt.show()

## Interpolant SS and Jacobians

In [2]:
model = DurableConsumer(**params)
t0 = time.time()
model.compute_steady_state()
print(
    f"SS interp: {model.completed_cycles} cycles, "
    f"{model.distance_metric}={model.distance:.3e}, {time.time()-t0:.1f}s"
)
display(pd.Series({
    "C": float(model.C_ss), "D": float(model.D_ss), "DEX": float(model.DEX_ss),
    "TEX": float(model.TEX_ss), "A": float(model.A_ss), "ADJ": float(model.ADJ_ss),
}, name="interp SS"))

t0 = time.time()
J_base = pack_jac(model.calc_jacobian("Rfree", T))
print(f"calc_jacobian(Rfree): {time.time()-t0:.1f}s")

t0 = time.time()
J_R = pack_jac(model.calc_jacobian_updatedR("Rfree", T))
print(f"calc_jacobian_updatedR(Rfree): {time.time()-t0:.1f}s")

print("max |J_updatedR - J_baseline|")
display(jac_diff_table(J_base, J_R).style.format("{:.4e}"))
plot_jac_triplet(J_base, J_R, r"Interpolant SS, $\partial / \partial Rfree$")

c:\Users\amonninger\AppData\Local\miniconda3\envs\DCEGM_Search_env\lib\site-packages\HARK\rewards.py:40: RuntimeWarning: divide by zero encountered in reciprocal
  return c ** (1.0 - rho) / (1.0 - rho)
c:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel.py:603: RuntimeWarning: divide by zero encountered in power
  * C ** (self.alpha * (1.0 - self.CRRA) - 1.0)
c:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel.py:610: RuntimeWarning: divide by zero encountered in power
  * (C ** (self.alpha * (1.0 - self.CRRA)))
c:\Users\amonninger\AppData\Local\miniconda3\envs\DCEGM_Search_env\lib\site-packages\HARK\rewards.py:40: RuntimeWarning: divide by zero encountered in reciprocal
  return c ** (1.0 - rho) / (1.0 - rho)
c:\Users\amonninger\AppData\Local\miniconda3\envs\DCEGM_Search_env\lib\site-packages\HARK\rewards.py:40: RuntimeWarning: divide by zero encountered in reciprocal
  return c ** (1.0 -

SS interp: 200 cycles, mse=1.794e-16, 60.5s


c:\Users\amonninger\AppData\Local\miniconda3\envs\DCEGM_Search_env\lib\site-packages\scipy\sparse\_index.py:151: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil_matrix is more efficient.
  self._set_arrayXarray(i, j, x)


C      0.824270
D      4.797361
DEX    0.239868
TEX    1.064138
A      1.225379
ADJ    0.128574
Name: interp SS, dtype: float64

c:\Users\amonninger\AppData\Local\miniconda3\envs\DCEGM_Search_env\lib\site-packages\HARK\rewards.py:40: RuntimeWarning: divide by zero encountered in reciprocal
  return c ** (1.0 - rho) / (1.0 - rho)
c:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel.py:610: RuntimeWarning: divide by zero encountered in power
  * (C ** (self.alpha * (1.0 - self.CRRA)))
c:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Code\DurableModel.py:603: RuntimeWarning: divide by zero encountered in power
  * C ** (self.alpha * (1.0 - self.CRRA) - 1.0)


KeyboardInterrupt: 

## Cond SS and Jacobians

In [ ]:
t0 = time.time()
model.compute_steady_state_cond()
print(f"SS cond: {time.time()-t0:.1f}s")
display(pd.Series({
    "C": float(model.C_ss), "D": float(model.D_ss), "DEX": float(model.DEX_ss),
    "TEX": float(model.TEX_ss), "A": float(model.A_ss), "ADJ": float(model.ADJ_ss),
}, name="cond SS"))

t0 = time.time()
J_base_c = pack_jac(model.calc_jacobian_cond("Rfree", T))
print(f"calc_jacobian_cond(Rfree): {time.time()-t0:.1f}s")

t0 = time.time()
J_R_c = pack_jac(model.calc_jacobian_updatedR_cond("Rfree", T))
print(f"calc_jacobian_updatedR_cond(Rfree): {time.time()-t0:.1f}s")

print("max |J_updatedR - J_baseline| (cond)")
display(jac_diff_table(J_base_c, J_R_c).style.format("{:.4e}"))
plot_jac_triplet(J_base_c, J_R_c, r"Cond SS, $\partial / \partial Rfree$")